# Multi-Agent Systems — AI Engineering Interview Prep

Supervisor pattern: orchestrator delegates tasks to specialized sub-agents.

**Prerequisites**: Set `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import anthropic
import os
import json
from typing import Any

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"

def call_claude(messages, system=None, max_tokens=400):
    kwargs = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[0].text

print(f"Multi-agent system with {MODEL}")

## 1. Specialized Sub-Agents

In [ ]:
# Each sub-agent has a specific expertise
class ResearchAgent:
    """Gathers and summarizes information about ML topics."""
    SYSTEM = "You are an ML research agent. Provide factual, concise summaries. Max 100 words."

    def run(self, task: str) -> str:
        return call_claude([{"role": "user", "content": task}], system=self.SYSTEM)


class CodeAgent:
    """Writes Python code for ML tasks."""
    SYSTEM = "You are an ML code agent. Write clean, runnable Python code with no explanations. Max 20 lines."

    def run(self, task: str) -> str:
        return call_claude([{"role": "user", "content": task}], system=self.SYSTEM)


class CriticAgent:
    """Reviews and critiques ML approaches."""
    SYSTEM = "You are an ML critic agent. Identify 3 potential issues or improvements. Be concise and specific."

    def run(self, task: str) -> str:
        return call_claude([{"role": "user", "content": task}], system=self.SYSTEM)


class WriterAgent:
    """Synthesizes findings into clear reports."""
    SYSTEM = "You are a technical writer. Synthesize the provided inputs into a clear, structured summary. Max 150 words."

    def run(self, task: str) -> str:
        return call_claude([{"role": "user", "content": task}], system=self.SYSTEM)


# Instantiate agents
agents = {
    "research": ResearchAgent(),
    "code": CodeAgent(),
    "critic": CriticAgent(),
    "writer": WriterAgent(),
}
print(f"Agents: {list(agents.keys())}")

## 2. Supervisor Agent

In [ ]:
SUPERVISOR_SYSTEM = """You are a supervisor agent coordinating a team of ML specialists.
Available agents: research, code, critic, writer.

For each user request, decide the sequence of agents to use.
Output ONLY valid JSON:
{"plan": [{"agent": "name", "task": "specific instruction for this agent"}]}

Rules:
- Use 'research' for background info and explanations
- Use 'code' for implementation tasks
- Use 'critic' when reviewing an approach or code
- Use 'writer' as the final step to synthesize results
- Limit to 3-4 agent calls per request"""

class SupervisorAgent:
    def __init__(self, agents: dict):
        self.agents = agents

    def plan(self, user_request: str) -> list:
        """Generate execution plan."""
        raw = call_claude(
            [{"role": "user", "content": f"Plan for: {user_request}"}],
            system=SUPERVISOR_SYSTEM,
            max_tokens=400
        )
        clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(clean).get("plan", [])

    def run(self, user_request: str) -> str:
        print(f"Supervisor received: {user_request}")
        print("=" * 60)

        plan = self.plan(user_request)
        results = {}

        for step in plan:
            agent_name = step["agent"]
            task = step["task"]
            # Inject previous results as context if available
            if results:
                context = "\n".join(f"[{k}]: {v[:200]}" for k, v in results.items())
                task = f"{task}\n\nContext from previous agents:\n{context}"

            print(f"[{agent_name.upper()}] Task: {step['task'][:80]}")
            result = self.agents[agent_name].run(task)
            results[agent_name] = result
            print(f"   Result: {result.strip()[:150]}")
            print()

        return results.get("writer", list(results.values())[-1])


supervisor = SupervisorAgent(agents)

# Test
final = supervisor.run("Explain transformer attention, provide a minimal Python implementation, and critique it.")
print("=" * 60)
print("FINAL OUTPUT:")
print(final)

## 3. Peer Review Pattern

In [ ]:
# Peer review: two agents independently produce outputs, third synthesizes
def peer_review_pipeline(question: str) -> dict:
    print(f"Question: {question}")

    # Two agents independently answer
    agent_a = ResearchAgent()
    agent_b = CodeAgent()

    answer_a = agent_a.run(f"Explain: {question}")
    answer_b = agent_b.run(f"Show with code example: {question}")

    # Critic reviews both
    critic = CriticAgent()
    critique = critic.run(f"Review these two responses:\nConceptual: {answer_a}\nCode: {answer_b}")

    # Writer synthesizes
    writer = WriterAgent()
    synthesis_task = f"""Synthesize into a complete answer:
Conceptual: {answer_a}
Code example: {answer_b}
Critique: {critique}"""
    synthesis = writer.run(synthesis_task)

    return {"conceptual": answer_a, "code": answer_b, "critique": critique, "synthesis": synthesis}


result = peer_review_pipeline("What is dropout and how does it prevent overfitting?")
print("\n=== SYNTHESIS ===")
print(result["synthesis"])

## 4. Multi-Agent Patterns Reference

In [ ]:
patterns = {
    "Supervisor/Worker": {
        "structure": "Orchestrator → delegates to specialists → collects results",
        "when": "Complex tasks requiring multiple skills",
        "tradeoff": "Orchestrator bottleneck; easier to debug"
    },
    "Peer Review": {
        "structure": "N agents produce outputs → critic compares → synthesizer merges",
        "when": "High-stakes decisions needing consensus",
        "tradeoff": "N× cost; reduces single-agent errors"
    },
    "Sequential Pipeline": {
        "structure": "Agent1 → Agent2 → Agent3 (output feeds next)",
        "when": "Linear workflows: research → draft → review → publish",
        "tradeoff": "Simple; errors propagate downstream"
    },
    "Parallel Fan-out": {
        "structure": "Task split → N agents run simultaneously → merge",
        "when": "Independent subtasks (e.g., research 5 papers)",
        "tradeoff": "Fast; requires result merging logic"
    },
    "Debate": {
        "structure": "Agent A proposes → Agent B argues against → judge decides",
        "when": "Adversarial validation, reducing hallucinations",
        "tradeoff": "High cost; stronger calibration"
    },
}

for pattern, info in patterns.items():
    print(f"\n{pattern}")
    for k, v in info.items():
        print(f"  {k}: {v}")

## Interview Tips

- **Supervisor pattern**: decouples task decomposition from execution. Supervisor can be LLM or rule-based.
- **Context passing**: each agent needs context from prior agents — pass relevant excerpts, not full outputs.
- **Agent granularity**: too coarse = bottleneck at one agent; too fine = coordination overhead.
- **Failure modes**: agent loops (A calls B calls A), context blowup, compounding errors.
- **Observability**: log each agent's input/output for debugging — treat as distributed system.
- **Cost**: N agents = N× API calls. Use parallelism only when tasks are truly independent.